In [ ]:
import kagglehub
import json
from PIL import Image
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [ ]:
# Load images

def get_images(download_dataset_flag):
    if download_dataset_flag:
        dataset_path = kagglehub.dataset_download("humansintheloop/teeth-segmentation-on-dental-x-ray-images")
        sources = {k: Path(dataset_path) / f'Teeth Segmentation {k}' for k in ['JSON', 'PNG']}
    else:
        dataset_root = Path(r"")
        sources = {k: dataset_root / f'./Teeth Segmentation {k}' for k in ['JSON', 'PNG']}

    meta = {}
    for p in sources.values():
        meta.update(json.loads((p / 'meta.json').read_text()))

    images = [
        (Image.open(img_path), img_path)
        for p in sources.values()
        for img_path in (p / 'd2' / 'img').glob('*')
    ]

    images_np = [(np.array(image), str(p)) for (image, p) in images[:len(images)//2]]
    return images_np

In [ ]:
DOWNLOAD_DATASET = False

images_np = get_images(DOWNLOAD_DATASET)

In [ ]:
# Get general info about images size

shapes = []
for image, p in images_np:
    shapes.append(image.shape)

min_width = np.min(np.array(shapes)[:, 1])
max_width = np.max(np.array(shapes)[:, 1])
min_height = np.min(np.array(shapes)[:, 0])
max_height = np.max(np.array(shapes)[:, 0])
avg_width = np.median(np.array(shapes)[:, 1])
avg_height = np.median(np.array(shapes)[:, 0])

print(min_width, max_width, min_height, max_height, avg_width, avg_height)


In [ ]:
# Method changes image size and calculates new polygon positions (but do not save them so we need to either run it every time of rewrite it to save new data

def normalize_image_size(image, image_metadata, output_height=1024, output_width=2045):
    image_height, image_width = image.shape

    scale_y = output_height / image_height
    scale_x = output_width / image_width

    resized_image = cv2.resize(image, (output_width, output_height))
    image_metadata['size'] = {
        'height': output_height,
        'width': output_width,
    }

    for segment_no in range(len(image_metadata['objects'])):
        for vertex_no in range(len(image_metadata['objects'][segment_no]['points']['exterior'])):
            x = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0]
            y = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1]
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0] = int(x*scale_x)
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1] = int(y*scale_y)

    return resized_image, image_metadata

In [ ]:
import copy

# Method transforms image_number raw images and metadata to resized images and metadata
def adjust_images(images_with_paths, image_number=None):
    normalized_images = []
    images_with_metadata = []
    for image, image_path in images_with_paths[:image_number if image_number else len(images_with_paths)]:
        metadata_filename = image_path.split('\\')[-1] + '.json'
        metadata_path = '\\'.join(image_path.split('\\')[:-2]) + '\\ann\\' + metadata_filename
        with open(metadata_path, 'r') as f:
            image_metadata = json.load(f)
        images_with_metadata.append((image, image_metadata, image_path))
        resized_image, resized_image_metadata = normalize_image_size(image, copy.deepcopy(image_metadata))
        normalized_images.append((resized_image, resized_image_metadata))

    return normalized_images, images_with_metadata

In [ ]:
def get_image_names(images_with_paths):
    return [p.split('\\')[-1] for _, p in images_with_paths]

In [ ]:
resized_images_np, original_images_np = adjust_images(images_np)

In [ ]:
# Compare how image and shapes created from metadata look before and after resize

def display_images_size_compared():
    i = 1
    ipo = 20
    for (resized_image, resized_metadata), (original_image, metadata, path) in zip(resized_images_np[i*ipo: (i+1)*ipo], original_images_np[i*ipo: (i+1)*ipo]):
        if original_image.shape[1] < 2100:
            print(path)
            original_xs = []
            original_ys = []
            for segment_no in range(len(metadata['objects'])):
                arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                original_xs.append(arr[:, 0])
                original_ys.append(arr[:, 1])

            resized_xs = []
            resized_ys = []
            for segment_no in range(len(resized_metadata['objects'])):
                arr = np.array(resized_metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(resized_metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                resized_xs.append(arr[:, 0])
                resized_ys.append(arr[:, 1])

            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            axes[1].imshow(resized_image, cmap='gray')
            axes[1].set_title(f'Resized image {resized_image.shape[0]}x{resized_image.shape[1]}')
            for x, y in zip(resized_xs, resized_ys):
                axes[1].plot(x, y)
            axes[1].axis('off')

            axes[0].imshow(original_image, cmap='gray')
            axes[0].set_title(f'Original image {original_image.shape[0]}x{original_image.shape[1]}')
            for x, y in zip(original_xs, original_ys):
                axes[0].plot(x, y)
            axes[0].axis('off')

            plt.show()

In [ ]:
display_images_size_compared()

In [ ]:
debug=False

In [ ]:
def extract_jaw_directional_edges(image):
    # Method detects multiple edges on the image, thanks to high clipLimit on CLAHE and very small threshold params on Canny method, then arctan between sobel mask edges in both directions, and try to keep only horizontal edges, for us to see more clearly the jawbone shape
    clahe = cv2.createCLAHE(clipLimit=15.0, tileGridSize=(4,4))
    enhanced = clahe.apply(image)
    if debug:
        plt.imshow(enhanced, cmap='gray')
        plt.show()
    blurred = cv2.GaussianBlur(enhanced, (3,3), 0)
    edges = cv2.Canny(blurred, threshold1=0, threshold2=20)
    if debug:
        plt.imshow(edges, cmap='gray')
        plt.show()

    sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=11)
    sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=11)
    edge_angle = np.arctan2(sobely, sobelx)

    h, w = image.shape
    center_x, center_y = w // 2, h // 2
    filtered_edges = np.zeros_like(edges)

    edge_coords = np.argwhere(edges > 0)

    for y, x in edge_coords:
        dx = x - center_x
        dy = y - center_y

        pixel_angle = np.arctan2(dy, dx)
        gradient_angle = edge_angle[y, x]
        angle_diff = abs(pixel_angle - gradient_angle)
        if angle_diff < np.pi / 2 or angle_diff > 3 * np.pi / 2:
            filtered_edges[y, x] = 255

    if debug:
        plt.imshow(filtered_edges, cmap='gray')
        plt.show()

    return filtered_edges

In [ ]:
def find_upper_mandible_border(idx, image, original):
    # Method finds horizontal lines without edges in the middle of the "Canny+Sobel" edged images, and for every horizontal line, check its difference to neighbouring (10 px lower) lines, if the difference is high, we set the line as a probable jawbone upper edge, use the line with the lowest position as a final jawbone upper edge
    h, w = image.shape
    image = cv2.GaussianBlur(image, (5,5), 0)
    if debug:
        plt.imshow(image, cmap='gray')
        plt.show()

    margin = int(w * 0.25)
    roi = image[:, margin:-margin]
    if debug:
        plt.imshow(roi, cmap='gray')
        plt.show()

    proj = np.sum(roi, axis=1).astype(np.float32)
    proj = cv2.GaussianBlur(proj.reshape(-1,1), (1,51), 0).flatten()
    search_from = int(h*0.02)
    search_to   = int(h*0.35)

    v=10
    ys = []
    diffs = []
    for y in range(search_from+v, search_to):
        diff = proj[y-v:y].mean() - proj[y:y+v].mean()
        ys.append(y)
        diffs.append(diff)

    diffs = np.array(diffs)
    ys = np.array(ys)
    max_diff = np.max(diffs)
    max_y = ys[np.argmax(diffs)]

    y_to_check = ys[diffs > (2500 if max_diff * 0.4 > 2500 else max_diff*0.4)]
    if debug:
        print(f"{idx}: Detected upper border at row: {max_y}, {max_diff}")

    vis = cv2.cvtColor(original, cv2.COLOR_GRAY2BGR)
    cv2.line(vis, (0, max_y), (w, max_y), (0, 255, 0), 3)
    for y in y_to_check:
        cv2.line(vis, (0, y), (w, y), (255, 0, 0), 3)
    cv2.line(vis, (0, int(h*0.4)), (w, int(h*0.4)), (0, 0, 255), 3)

    return vis, y_to_check[-1]

In [ ]:
top_crop_images = []
# Crop every image (10% from the bottom and some part from the top)
for idx in range(len(resized_images_np)):
    original_img, metadata = resized_images_np[idx]
    h, w = original_img.shape

    original_img = original_img[0:int(h*0.9), :]

    edge_img = extract_jaw_directional_edges(original_img)
    result_img, top_y_bound = find_upper_mandible_border(idx, edge_img, original_img)

    top_crop_image = original_img[top_y_bound:, :]

    top_crop_images.append((top_crop_image, (metadata, top_y_bound)))

In [ ]:
def create_bone_mask(image):
    # Find the jawbone on the cropped image, adaptive thresholding, then erosion of the left and right 15% of the image, to separate jawbone from the noise and light on the image, then find contours on the image, and get all that are bigger than some threshold = 30000 of its area, then combine all the chosen edges, create convex hull around them, and its shape is the mask that contains jawbone
    binary = cv2.adaptiveThreshold(
        image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=1501,
        C=10
    )
    if debug:
        plt.imshow(binary, cmap='gray')
        plt.show()

    w,h = binary.shape
    part = 0.15
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (23,23))
    binary[:, :int(w*part)] = cv2.morphologyEx(binary[:, :int(w*part)], cv2.MORPH_ERODE, kernel, iterations=6)
    binary[:, -int(w*part):] = cv2.morphologyEx(binary[:, -int(w*part):], cv2.MORPH_ERODE, kernel, iterations=6)

    if debug:
        plt.imshow(binary, cmap='gray')
        plt.show()

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return np.ones_like(image) * 255

    jaw_contours = sorted(contours, key=cv2.contourArea, reverse=True)
    jaw_mask_before = np.zeros_like(image)
    max_contour = 30_000
    i=0
    chosen_contours = []
    while cv2.contourArea(jaw_contours[i]) >= max_contour:
        cv2.drawContours(jaw_mask_before, [jaw_contours[i]], -1, 255, -1)
        chosen_contours.append(jaw_contours[i])
        i+=1
    if debug:
        print(np.unique(jaw_mask_before))
        plt.imshow(jaw_mask_before, cmap='gray')
        plt.show()

    h, w = jaw_mask_before.shape
    combined_points = np.vstack(chosen_contours)
    boundary = cv2.convexHull(combined_points)

    if debug:
        vis = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        for contour in chosen_contours:
            cv2.drawContours(vis, [contour], -1, (0, 255, 0), 2)
        cv2.drawContours(vis, [boundary], -1, (255, 255, 0), 3)
        plt.imshow(vis, cmap='gray')
        plt.show()

    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(mask, [boundary], -1, 255, -1)

    if debug:
        plt.imshow(mask, cmap='gray')
        plt.show()

    return mask

def apply_bone_mask(image, mask):
    # Use mask on the xray image
    return cv2.bitwise_and(image, image, mask=mask)


In [ ]:
def update_image_metadata(masked_image, resized_metadata):
    image_height, image_width = masked_image.shape
    metadata, top_y = resized_metadata
    metadata = copy.deepcopy(metadata)
    metadata['size'] = {
        'height': image_height,
        'width': image_width,
    }

    for segment_no in range(len(metadata['objects'])):
        for vertex_no in range(len(metadata['objects'][segment_no]['points']['exterior'])):
            y = metadata['objects'][segment_no]['points']['exterior'][vertex_no][1]
            y -= top_y
            if y < 0:
                y = 0
            if y > image_height-1:
                y = image_height-1
            metadata['objects'][segment_no]['points']['exterior'][vertex_no][1] = y

    return metadata

In [ ]:
masked_images = []
for idx in range(len(resized_images_np)):
    original_img, metadata = top_crop_images[idx]
    h, w = original_img.shape

    denoised = cv2.bilateralFilter(original_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(original_img, bone_mask)
    # plt.imshow(teeth_region, cmap='gray')
    # plt.show()
    masked_metadata = update_image_metadata(teeth_region, metadata)
    masked_images.append(((bone_mask, teeth_region), masked_metadata))


In [ ]:
#  Display ground truth contours on the cropped + jawbone masked images

def display_contours_on_masked_image(masked_images_data, image_idxs):
    for ((_, masked_image), metadata), image_idx in zip(masked_images_data, image_idxs):
        xs = []
        ys = []
        for segment_no in range(len(metadata['objects'])):
            arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
            arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
            xs.append(arr[:, 0])
            ys.append(arr[:, 1])

        plt.imshow(masked_image, cmap='gray')
        plt.title(f'Image {image_idx}')
        for x, y in zip(xs, ys):
            plt.plot(x, y)
        plt.axis('off')

        plt.show()

In [ ]:
idxs = sorted([str(i) for i in range(1,len(resized_images_np)+1)])
display_contours_on_masked_image(masked_images[:20], idxs[:20])

In [ ]:
teeth_class_to_teeth_type_map = {}
for i in range(1,33):
    if i in [1,2,3,14,15,16,17,18,19,30,31,32]:
        teeth_class_to_teeth_type_map[i] = 0
    elif i in [4,5,12,13,20,21,28,29]:
        teeth_class_to_teeth_type_map[i] = 1
    elif i in [6,11,22,27]:
        teeth_class_to_teeth_type_map[i] = 2
    elif i in [7,8,9,10,23,24,25,26]:
        teeth_class_to_teeth_type_map[i] = 3

In [ ]:
#  Method to seperate xrays of children teeth and some corrupted data (multiple contours for teeth of the same class, we discard 64 photos here)

def discard_children_teeth_xrays(one_masked_image_data, image_idx):
    (_, masked_image), metadata = one_masked_image_data
    xs = []
    ys = []
    teeth_classes = []
    teeth_types = []
    for segment_no in range(len(metadata['objects'])):
        arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
        arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
        teeth_classes.append(int(metadata['objects'][segment_no]['classTitle']))
        teeth_types.append(teeth_class_to_teeth_type_map[teeth_classes[-1]])
        xs.append(arr[:, 0])
        ys.append(arr[:, 1])

    teeth_type_to_colour_map = {
        0: 'red',
        1: 'blue',
        2: 'green',
        3: 'yellow',
    }

    if len(set(teeth_classes)) != len(teeth_classes) != 0:
        plt.imshow(masked_image, cmap='gray')
        plt.title(f'Image {image_idx}')
        print(sorted(teeth_classes))
        for x, y, teeth_type in zip(xs, ys, teeth_types):
            plt.plot(x, y,color=teeth_type_to_colour_map[teeth_type])
        plt.axis('off')

        plt.show()

        return False
    return True

In [ ]:
masked_images_adults = []
idxs_adults = []
masked_images_children = []
idxs_children = []
for masked_image_data, idx in zip(masked_images, idxs):
    result = discard_children_teeth_xrays(masked_image_data, idx)
    if result:
        masked_images_adults.append(masked_image_data)
        idxs_adults.append(idx)
    else:
        masked_images_children.append(masked_image_data)
        idxs_children.append(idx)

In [ ]:
print(len(masked_images_adults), len(masked_images_children))

In [ ]:
# Get shape features / descriptors from single contour

def get_contour_features(contour):
    features = {}
    features['area'] = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)

    # 3. Bounding box dimensions
    x, y, w, h = cv2.boundingRect(contour)
    features['bbox_width'] = w
    features['bbox_height'] = h
    features['bbox_area'] = w * h

    features['aspect_ratio'] = h / w if w > 0 else 0

    hull = cv2.convexHull(contour)
    # 7. Convexity (convex hull perimeter / contour perimeter)
    hull_perimeter = cv2.arcLength(hull, True)
    features['convexity'] = hull_perimeter / perimeter

    # 9. Circularity (4π * area / perimeter^2)
    # Inverse of compactness, circle = 1.0
    features['circularity'] = (4 * np.pi * features['area']) / (perimeter ** 2)

    if len(contour) >= 5:
        ellipse = cv2.fitEllipse(contour)
        center, axes, angle = ellipse
        major_axis = max(axes)
        minor_axis = min(axes)

        features['ellipse_minor_axis'] = minor_axis
        features['ellipse_eccentricity'] = np.sqrt(1 - (minor_axis/major_axis)**2) if major_axis > 0 else 0
    else:
        return None

    _, radius = cv2.minEnclosingCircle(contour)
    circle_area = np.pi * (radius ** 2)
    features['circle_occupancy'] = features['area'] / circle_area if circle_area > 0 else 0

    moments = cv2.moments(contour)
    hu_moments = cv2.HuMoments(moments).flatten()
    hu_moments_log = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)

    for i, hu in enumerate(hu_moments_log[:2]):
        features[f'hu_moment_{i+1}'] = hu

    mask = np.zeros(image.shape, dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1)

    # Extract tooth region
    tooth_pixels = image[mask == 255]
    features['intensity_mean'] = np.mean(tooth_pixels)

    return features


In [ ]:
from collections import defaultdict

#  Method to get teeth features from ground truth segments, later to be replaced with contours given by segmentation method

def get_teeth_features(one_masked_image_data):
    (_, masked_image), metadata = one_masked_image_data
    teeth_classes = []
    teeth_types = []
    teeth_features = defaultdict(lambda: list())
    for segment_no in range(len(metadata['objects'])):
        arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
        arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
        teeth_classes.append(int(metadata['objects'][segment_no]['classTitle']))
        teeth_contour = np.stack([arr[:, 0], arr[:, 1]], axis=1)
        teeth_contour = teeth_contour.reshape(-1, 1, 2)
        segment_features = get_contour_features(teeth_contour)
        if segment_features:
            for k, v in segment_features.items():
                teeth_features[k].append(v)
            teeth_types.append(teeth_class_to_teeth_type_map[teeth_classes[-1]])

    return teeth_features, teeth_types

In [ ]:
all_images_features = defaultdict(lambda: list())
all_images_teeth_types = list()
for masked_image_data, idx in zip(masked_images_adults, idxs_adults):
    one_image_features, one_image_teeth_types = get_teeth_features(masked_image_data)
    for k,v in one_image_features.items():
        all_images_features[k].extend(v)
    all_images_teeth_types.extend(one_image_teeth_types)

for k,v in all_images_features.items():
    print(k, len(v))

In [ ]:
from scipy import stats
import pandas as pd


# Method to find out which features from shape and texture descriptors are the most unifed in every tooth type and distinguishable between different one

def analyze_feature_discriminability(all_features, all_labels, debug=True):
    df = pd.DataFrame(all_features)
    df['tooth_type'] = all_labels

    feature_scores = []
    for feature in df.columns:
        if feature == 'tooth_type':
            continue

        groups = [df[df['tooth_type'] == t][feature].values for t in range(4)]
        f_stat, p_value = stats.f_oneway(*groups)

        # === METRIC 2: Between-class variance / Within-class variance ===
        # Fisher's criterion
        overall_mean = df[feature].mean()
        between_var = 0
        for t in range(4):
            group = df[df['tooth_type'] == t][feature]
            class_mean = group.mean()
            n_samples = len(group)
            between_var += n_samples * (class_mean - overall_mean) ** 2
        between_var /= len(df)

        within_var = 0
        for t in range(4):
            group = df[df['tooth_type'] == t][feature]
            within_var += group.var() * len(group)
        within_var /= len(df)

        fisher_score = between_var / within_var if within_var > 0 else 0

        # === METRIC 3: Coefficient of Variation Ratio ===
        # Compare variation BETWEEN classes vs variation WITHIN classes

        class_means = [df[df['tooth_type'] == t][feature].mean() for t in range(4)]
        cv_between = np.std(class_means) / np.mean(class_means)
        cv_within = np.mean([df[df['tooth_type'] == t][feature].std() / df[df['tooth_type'] == t][feature].mean() for t in range(4)])
        cv_ratio = cv_between / cv_within

        # === METRIC 4: Class separation (min distance between means) ===
        means = {t: df[df['tooth_type'] == t][feature].mean() for t in range(4)}
        mean_values = list(means.values())
        min_separation = min([abs(mean_values[i] - mean_values[j]) for i in range(len(mean_values)) for j in range(i+1, len(mean_values))])

        ranges = {t: (df[df['tooth_type'] == t][feature].min(), df[df['tooth_type'] == t][feature].max()) for t in range(4)}
        overlaps = []
        for t1 in range(4):
            for t2 in range(4):
                if t1 >= t2:
                    continue

                min1, max1 = ranges[t1]
                min2, max2 = ranges[t2]
                overlap = max(0, min(max1, max2) - max(min1, min2))

                range1 = max1 - min1
                range2 = max2 - min2
                overlap_frac = overlap / min(range1, range2)

                overlaps.append(overlap_frac)
        avg_overlap = np.mean(overlaps) if overlaps else 0

        feature_scores.append({
            'feature': feature,
            'f_statistic': f_stat,
            'p_value': p_value,
            'fisher_score': fisher_score,
            'cv_ratio': cv_ratio,
            'min_separation': min_separation,
            'avg_overlap': avg_overlap,
            'overall_score': fisher_score * (1 - avg_overlap)  # Combined metric
        })
    scores_df = pd.DataFrame(feature_scores)
    scores_df = scores_df.sort_values('overall_score', ascending=False)

    if debug:
        print("="*80)
        print("FEATURE DISCRIMINABILITY ANALYSIS")
        print("="*80)
        print("\nTop 15 most discriminative features:")
        print(scores_df.to_string(index=False))

        print("\n\nINTERPRETATION:")
        print("- f_statistic: Higher is better (>10 is good)")
        print("- fisher_score: Higher is better (>1 is good)")
        print("- cv_ratio: Higher is better (>1 means between-class variation > within-class)")
        print("- avg_overlap: Lower is better (0 = no overlap, 1 = complete overlap)")
        print("- overall_score: Combined metric (higher is better)")

    return scores_df, df

In [ ]:
def visualize_feature_distributions(df, feature_scores, top_n=12):
    top_features = feature_scores.head(top_n)['feature'].values

    n_cols = 3
    n_rows = (top_n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
    axes = axes.flatten()
    teeth_type_to_colour_map = {
        0: 'red',
        1: 'blue',
        2: 'green',
        3: 'yellow',
    }
    teeth_type_to_type_name = {
        0: 'molar',
        1: 'premolar',
        2: 'canine',
        3: 'incisor'
    }
    for idx, feature in enumerate(top_features):
        ax = axes[idx]

        for tooth_type in range(4):
            data = df[df['tooth_type'] == tooth_type][feature].dropna()

            if len(data) > 0:
                ax.hist(data, alpha=0.5, label=teeth_type_to_type_name[tooth_type],
                       color=teeth_type_to_colour_map[tooth_type],
                       bins=20, density=True)

        ax.set_title(f'{feature}\n(Score: {feature_scores[feature_scores["feature"]==feature]["overall_score"].values[0]:.2f})')
        ax.set_xlabel('Value')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_feature_boxplots(df, feature_scores, top_n=12):
    top_features = feature_scores.head(top_n)['feature'].values

    n_cols = 3
    n_rows = (top_n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
    axes = axes.flatten()
    teeth_type_to_type_name = {
        0: 'molar',
        1: 'premolar',
        2: 'canine',
        3: 'incisor'
    }
    teeth_type_to_colour_map = {
        0: 'red',
        1: 'blue',
        2: 'green',
        3: 'yellow',
    }
    for idx, feature in enumerate(top_features):
        ax = axes[idx]

        data_to_plot = [df[df['tooth_type'] == t][feature].dropna().values for t in range(4)]
        bp = ax.boxplot(data_to_plot, tick_labels=list(teeth_type_to_type_name.values()), patch_artist=True)

        for patch, color in zip(bp['boxes'], list(teeth_type_to_colour_map.values())):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)

        ax.set_title(f'{feature}\n(F-stat: {feature_scores[feature_scores["feature"]==feature]["f_statistic"].values[0]:.1f})')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('feature_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def analyze_pairwise_separation(df, feature):
    print(f"\nPAIRWISE SEPARATION FOR: {feature}")
    print("="*80)
    teeth_type_to_type_name = {
        0: 'molar',
        1: 'premolar',
        2: 'canine',
        3: 'incisor'
    }
    separation_matrix = np.zeros((4, 4))

    for i in range(4):
        for j in range(4):
            if i >= j:
                continue

            data1 = df[df['tooth_type'] == i][feature].dropna()
            data2 = df[df['tooth_type'] == j][feature].dropna()

            # T-test (are means significantly different?)
            t_stat, p_value = stats.ttest_ind(data1, data2)

            # Effect size (Cohen's d)
            pooled_std = np.sqrt((data1.std()**2 + data2.std()**2) / 2)
            cohens_d = abs(data1.mean() - data2.mean()) / pooled_std if pooled_std > 0 else 0

            # Overlap coefficient
            min1, max1 = data1.min(), data1.max()
            min2, max2 = data2.min(), data2.max()
            overlap = max(0, min(max1, max2) - max(min1, min2))
            overlap_frac = overlap / min(max1 - min1, max2 - min2) if min(max1 - min1, max2 - min2) > 0 else 0

            separation_matrix[i, j] = cohens_d

            # Interpretation
            if cohens_d > 1.0:
                verdict = "✓✓ EXCELLENT"
            elif cohens_d > 0.5:
                verdict = "✓ GOOD"
            elif cohens_d > 0.2:
                verdict = "~ MODERATE"
            else:
                verdict = "✗ POOR"

            print(f"{teeth_type_to_type_name[i]:10s} vs {teeth_type_to_type_name[j]:10s}: "
                  f"Cohen's d={cohens_d:.3f}  "
                  f"overlap={overlap_frac:.2f}  "
                  f"p-value={p_value:.4f}  "
                  f"{verdict}")

    return separation_matrix

In [ ]:
def create_statistical_summary(df, feature_scores, top_n=12):
    top_features = feature_scores.head(top_n)['feature'].values
    summary_data = []
    teeth_type_to_type_name = {
        0: 'molar',
        1: 'premolar',
        2: 'canine',
        3: 'incisor'
    }
    for feature in top_features:
        row = {'Feature': feature}

        for tooth_type in range(4):
            data = df[df['tooth_type'] == tooth_type][feature].dropna()
            row[f'{teeth_type_to_type_name[tooth_type]}_mean'] = data.mean()
            row[f'{teeth_type_to_type_name[tooth_type]}_std'] = data.std()
            row[f'{teeth_type_to_type_name[tooth_type]}_median'] = data.median()

        summary_data.append(row)

    summary_df = pd.DataFrame(summary_data)

    print("\n" + "="*120)
    print("STATISTICAL SUMMARY - TOP FEATURES")
    print("="*120)

    # Print with formatting
    for feature in top_features:
        print(f"\n{feature}:")
        print("-" * 100)

        for tooth_type in range(4):
            data = df[df['tooth_type'] == tooth_type][feature].dropna()

            print(f"  {teeth_type_to_type_name[tooth_type]:12s}: "
                  f"mean={data.mean():8.3f}  "
                  f"std={data.std():8.3f}  "
                  f"median={data.median():8.3f}  "
                  f"range=[{data.min():7.3f}, {data.max():7.3f}]  "
                  f"n={len(data):4d}")

        means = [df[df['tooth_type'] == t][feature].mean() for t in range(4)]

        mean_range = max(means) - min(means)
        avg_std = np.mean([df[df['tooth_type'] == t][feature].std() for t in range(4)])
        separation_ratio = mean_range / avg_std if avg_std > 0 else 0

        print(f"  {'SEPARATION':12s}: "
              f"mean_range={mean_range:.3f}  "
              f"avg_std={avg_std:.3f}  "
              f"ratio={separation_ratio:.2f} "
              f"{'✓ GOOD' if separation_ratio > 2 else '✗ WEAK'}")

    return summary_df


In [ ]:
feature_scores, feature_data = analyze_feature_discriminability(all_images_features, all_images_teeth_types, debug=True)
# visualize_feature_distributions(feature_data, feature_scores, top_n=45)
# plot_feature_boxplots(feature_data, feature_scores, top_n=45)

In [ ]:
feature_data.head()

In [ ]:
feature_data.drop('tooth_type', axis=1, inplace=True)
feature_data_np = feature_data.to_numpy()

In [ ]:
feature_data_np.shape

In [ ]:
import pickle
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(feature_data_np, all_images_teeth_types, random_state=42, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# The one
classifier = SVC(kernel='rbf', C=50, gamma='auto')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(f"SVM accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}")
with open('teeth_type_svm.pkl','wb') as f:
    pickle.dump(classifier,f)
#
# clf = RandomForestClassifier(max_depth=15, n_estimators=500, max_features=3, criterion='entropy', random_state=42)
# clf.fit(X_train, y_train)
# y_pred = clf.predict(X_test)
# print(f'RFC accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}')
#
# km = KMeans(n_clusters=4, random_state=42)
# km.fit(X_train, y_train)
# y_pred = km.predict(X_test)
# print(f'KM accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

def remove_correlated_features(df, features, threshold=0.85, debug=True):
    """
    Remove highly correlated features
    Keep the one with better discriminative score
    """
    # Get correlation matrix
    corr_matrix = df.corr().abs()

    if debug:
        # Visualize correlation matrix
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                   square=True, cbar_kws={'label': 'Correlation'})
        plt.title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.savefig('correlation_matrix.png', dpi=150)
        plt.show()

    # Find pairs with correlation > threshold
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if corr_matrix.iloc[i, j] > threshold:
                high_corr_pairs.append((corr_matrix.columns[i],
                                       corr_matrix.columns[j],
                                       corr_matrix.iloc[i, j]))

    if debug and high_corr_pairs:
        print(f"\nFound {len(high_corr_pairs)} highly correlated pairs (>{threshold}):")
        for f1, f2, corr in high_corr_pairs:
            print(f"  {f1:30s} <-> {f2:30s}  (r={corr:.3f})")

    # Decide which to remove (assume you have feature_scores from before)
    # Keep the feature with better discriminative score
    features_to_remove = set()

    for f1, f2, corr in high_corr_pairs:
        # Get scores (if available)
        # Otherwise, remove the second one by default
        if f1 in features_to_remove:
            continue
        if f2 in features_to_remove:
            continue

        # Remove the one with lower variance (less informative)
        if df[f1].std() < df[f2].std():
            features_to_remove.add(f1)
            if debug:
                print(f"  → Removing {f1} (lower std)")
        else:
            features_to_remove.add(f2)
            if debug:
                print(f"  → Removing {f2} (lower std)")

    # Final feature set
    final_features = [f for f in features if f not in features_to_remove]

    if debug:
        print(f"\nOriginal features: {len(features)}")
        print(f"Removed features: {len(features_to_remove)}")
        print(f"Final features: {len(final_features)}")

    return final_features, list(features_to_remove)


In [ ]:
selected_features, removed_features = remove_correlated_features(
    feature_data,
    list(feature_data),
    threshold=0.85,
    debug=True
)

In [ ]:
selected_df1 = feature_data[selected_features]
print(selected_df1)

In [ ]:
feature_data_np8 = selected_df1.to_numpy()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(feature_data_np8, all_images_teeth_types, random_state=42, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

classifier = SVC(kernel='rbf', C=50, gamma='auto')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(f"SVM accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}")

clf = RandomForestClassifier(max_depth=15, n_estimators=500, max_features=3, criterion='entropy', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f'RFC accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}')

km = KMeans(n_clusters=4, random_state=42)
km.fit(X_train, y_train)
y_pred = km.predict(X_test)
print(f'KM accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}')

In [ ]:
from sklearn.model_selection import cross_val_score

from sklearn.feature_selection import RFECV


def recursive_feature_elimination(df, features, labels, debug=True):
    X = df[features].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Method A: RFECV (cross-validated, finds optimal number automatically)
    clf = RandomForestClassifier(max_depth=15, n_estimators=100, max_features=3, criterion='entropy', random_state=42)

    rfecv = RFECV(estimator=clf, step=1, cv=5, scoring='accuracy', n_jobs=-1)
    rfecv.fit(X_scaled, labels)

    # Get selected features
    selected_mask = rfecv.support_
    selected_features = [features[i] for i, selected in enumerate(selected_mask) if selected]

    if debug:
        print("="*80)
        print("RECURSIVE FEATURE ELIMINATION (RFE)")
        print("="*80)
        print(f"\nOptimal number of features: {rfecv.n_features_}")
        print(f"Best cross-validation score: {rfecv.cv_results_['mean_test_score'].max():.3f}")

        # Plot number of features vs accuracy
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
                rfecv.cv_results_['mean_test_score'], marker='o')
        plt.xlabel('Number of features')
        plt.ylabel('Cross-validation accuracy')
        plt.title('RFE: Accuracy vs Number of Features')
        plt.grid(True, alpha=0.3)
        plt.axvline(rfecv.n_features_, color='red', linestyle='--',
                   label=f'Optimal: {rfecv.n_features_} features')
        plt.legend()
        plt.tight_layout()
        plt.savefig('rfe_curve.png', dpi=150)
        plt.show()

        print(f"\nSelected {len(selected_features)} features:")
        for i, feat in enumerate(selected_features, 1):
            print(f"  {i}. {feat}")

        # Test accuracy with selected features
        X_selected = df[selected_features].values
        X_selected_scaled = scaler.fit_transform(X_selected)
        scores = cross_val_score(clf, X_selected_scaled, labels, cv=5)
        print(f"\nAccuracy with selected features: {scores.mean():.3f} ± {scores.std():.3f}")

    return selected_features, rfecv

In [ ]:
recursive_feature_elimination(feature_data,
    list(feature_data),
    all_images_teeth_types,
    debug=True)

In [ ]:
from sklearn.model_selection import cross_val_score

from sklearn.feature_selection import RFECV


def recursive_feature_elimination(df, features, labels, debug=True):
    X = df[features].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Method A: RFECV (cross-validated, finds optimal number automatically)
    clf = SVC(kernel='linear', C=50, gamma='auto')

    rfecv = RFECV(estimator=clf, step=1, cv=5, scoring='accuracy', n_jobs=-1)
    rfecv.fit(X_scaled, labels)

    # Get selected features
    selected_mask = rfecv.support_
    selected_features = [features[i] for i, selected in enumerate(selected_mask) if selected]

    if debug:
        print("="*80)
        print("RECURSIVE FEATURE ELIMINATION (RFE)")
        print("="*80)
        print(f"\nOptimal number of features: {rfecv.n_features_}")
        print(f"Best cross-validation score: {rfecv.cv_results_['mean_test_score'].max():.3f}")

        # Plot number of features vs accuracy
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
                rfecv.cv_results_['mean_test_score'], marker='o')
        plt.xlabel('Number of features')
        plt.ylabel('Cross-validation accuracy')
        plt.title('RFE: Accuracy vs Number of Features')
        plt.grid(True, alpha=0.3)
        plt.axvline(rfecv.n_features_, color='red', linestyle='--',
                   label=f'Optimal: {rfecv.n_features_} features')
        plt.legend()
        plt.tight_layout()
        plt.savefig('rfe_curve.png', dpi=150)
        plt.show()

        print(f"\nSelected {len(selected_features)} features:")
        for i, feat in enumerate(selected_features, 1):
            print(f"  {i}. {feat}")

        # Test accuracy with selected features
        X_selected = df[selected_features].values
        X_selected_scaled = scaler.fit_transform(X_selected)
        scores = cross_val_score(clf, X_selected_scaled, labels, cv=5)
        print(f"\nAccuracy with selected features: {scores.mean():.3f} ± {scores.std():.3f}")

    return selected_features, rfecv

In [ ]:
recursive_feature_elimination(feature_data,
    list(feature_data),
    all_images_teeth_types,
    debug=True)

In [ ]:
def sequential_feature_selection(df, features, labels, debug=True):
    """
    Sequential feature selection
    - Forward: Start with 0, add best feature iteratively
    - Backward: Start with all, remove worst feature iteratively
    """
    X = df[features].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf = SVC(kernel='rbf', C=50, gamma='auto')

    best_score = 0
    best_n = 0
    best_features = []

    for n in range(2, len(features), 2):
        sfs = SequentialFeatureSelector(
            clf,
            n_features_to_select=n,
            direction='forward',
            cv=5,
            n_jobs=-1
        )
        sfs.fit(X_scaled, labels)

        selected_mask = sfs.get_support()
        selected = [features[i] for i, sel in enumerate(selected_mask) if sel]

        # Test accuracy
        X_sel = df[selected].values
        X_sel_scaled = scaler.fit_transform(X_sel)
        score = cross_val_score(clf, X_sel_scaled, labels, cv=5).mean()

        print(f"n={n:2d}: score={score:.3f}")

        if score > best_score:
            best_score = score
            best_n = n
            best_features = selected

    if debug:
        print(f"\nBest number of features: {best_n}")
        print(f"Best accuracy: {best_score:.3f}")
        print(f"Selected features: {best_features}")

    return best_features, best_score


In [ ]:
forward_features, forward_score = sequential_feature_selection(
    feature_data,
    list(feature_data),
    all_images_teeth_types,
    debug=True
)

In [ ]:
n_estimators_list = [100,500]
max_features_list = [3,5,7]
criterion_list = ['gini', 'entropy', 'log_loss']
max_depth_list = [7,10,15]

for n_estimators in n_estimators_list:
    for max_features in max_features_list:
        for criterion in criterion_list:
            for max_depth in max_depth_list:
                clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, criterion=criterion, max_depth=max_depth, random_state=42)
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
                print(f'RFC (n_estimators={n_estimators}, max_features={max_features}, criterion={criterion}, max_depth={max_depth}) accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}')

In [ ]:
kernels=('linear', 'rbf', 'poly')
cs=[0.1, 1, 5, 10, 25, 50, 100, 500]
gammas=['scale', 'auto', 1, 0.1, 0.01]

for kernel in kernels:
    for c in cs:
        for gamma in gammas:
            classifier = SVC(kernel=kernel, C=c, gamma=gamma, random_state=42)
            classifier.fit(X_train, y_train)
            y_pred = classifier.predict(X_test)
            print(f"SVM (kernel={kernel}, C={c}, gamma={gamma}) accuracy on ground truth: {accuracy_score(y_test, y_pred):.3f}")

In [ ]:
def display_type_classification_compared(model, scaler, masked_images_data, image_idxs):
    (_, masked_image), metadata = masked_image_data
    one_image_features, one_image_teeth_types = get_teeth_features(masked_image_data)
    one_image_features = np.array(list(one_image_features.values())).T

    one_image_features = scaler.transform(one_image_features)
    predicted_teeth_types = model.predict(one_image_features)
    xs = []
    ys = []
    teeth_classes = []
    teeth_types = []
    for segment_no in range(len(metadata['objects'])):
        arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
        arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
        teeth_classes.append(int(metadata['objects'][segment_no]['classTitle']))
        teeth_types.append(teeth_class_to_teeth_type_map[teeth_classes[-1]])
        xs.append(arr[:, 0])
        ys.append(arr[:, 1])

    teeth_type_to_colour_map = {
        0: 'red',
        1: 'blue',
        2: 'green',
        3: 'yellow',
    }

    fig, axes = plt.subplots(1, 2, figsize=(10,20))

    axes[0].imshow(masked_image, cmap='gray')
    axes[0].set_title(f'Image {idx} original teeth types')
    for x, y, teeth_type in zip(xs, ys, teeth_types):
        axes[0].plot(x, y,color=teeth_type_to_colour_map[teeth_type])
    axes[0].axis('off')

    axes[1].imshow(masked_image, cmap='gray')
    axes[1].set_title(f'Image {idx} predicted teeth types')
    for x, y, teeth_type in zip(xs, ys, predicted_teeth_types):
        axes[1].plot(x, y,color=teeth_type_to_colour_map[teeth_type])
    axes[1].axis('off')

    plt.show()

In [ ]:
for masked_image_data, idx in zip(masked_images_adults[:5], idxs_adults):
    display_type_classification_compared(clf, scaler, masked_image_data, idx)

In [ ]:
summary = create_statistical_summary(feature_data, feature_scores, top_n=45)

In [ ]:
print("\n" + "="*80)
print("DETAILED PAIRWISE ANALYSIS")
print("="*80)
for feature in feature_scores['feature']:
    analyze_pairwise_separation(feature_data, feature)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

idxs_train, idxs_test, y_train, y_test = train_test_split(idxs_adults, teeth_types_labels)

for features in possible_features_sets:
    X_train = features[idxs_train]
    X_test = features[idxs_test]
    classifier = SVC(kernel='rbf')
    classifier.fit(X_train, y_train)

    print(f"SVM accuracy on ground truth: {classifier.score(X_test, y_test)}")

    clf = RandomForestClassifier(max_depth=10, n_estimators=250, max_features=10, criterion='gini', random_state=42)
    clf.fit(X_train, y_train)
    print(f'RFC accuracy on ground truth: {clf.score(X_test, y_test)}')

    km = KMeans(n_clusters=4, random_state=42)
    km.fit(X_train, y_train)
    print(f'KM accuracy on ground truth: {clf.score(X_test, y_test)}')